# Lesson 2: Anatomy of an eval

Companion to `curriculum/02-anatomy-of-an-eval.md`.

**Big idea:** every eval has the same four parts: **dataset → task → grader → metric**. Once you can see
those parts, you can build, debug, and critique any eval.

Mock model only: free, offline, no API key. Run cells top to bottom.
**🧠 Check** = answer in your own words before moving on. **✍️ Your turn** = cells for you to write (ask Claude for a hint).

> Your thread through the course: an AI **validator** that reads an Issue from the GRC platform and checks
> its rating against the Incident & Issue Management Standard.

In [ ]:
import json, random, re
from collections import Counter, defaultdict

from evalkit import app, graders
from evalkit.stats import bootstrap_ci, mean, paired_bootstrap_delta, wilson, cohens_kappa
from nb_helpers import DEV, HOLDOUT, load, run, report, failures, mock_task

dev = load(DEV)
holdout = load(HOLDOUT)
print(f"dev={len(dev)} cases, holdout={len(holdout)} cases")

```
 dataset (cases)        task (system under test)       grader(s)          metric(s)
 ┌──────────────┐       ┌──────────────────────┐      ┌───────────┐     ┌──────────────┐
 │ input        │──────▶│ prompt + model +     │─────▶│ score each │────▶│ aggregate    │
 │ expected     │       │ tools/context        │ out  │ output     │     │ across cases │
 │ tags, notes  │       └──────────────────────┘      └───────────┘     └──────────────┘
 └──────────────┘
```

## 1. Cases

A good case has an **input** (exactly what production sees), an **expected** answer, **acceptable
alternatives** where experts reasonably disagree, **tags** for slicing, and ideally **notes** on why it exists.
Look at `t006`. Its expected value allows two priorities:

In [ ]:
t006 = next(c for c in dev if c["id"] == "t006")
print(json.dumps(t006, indent=2))

### 🧠 Check
Why is an `acceptable` list better than forcing one "right" priority for `t006`? When would it be a cop-out?
(Think: two risk reviewers who'd rate the same Issue *Medium* vs *High*.)

## 2. Task → graders

Each grader checks **one** property. Single-purpose graders make failures easy to diagnose.

In [ ]:
case = dev[5]
out = mock_task(case["input"], seed=0)
print("input:   ", case["input"])
print("output:  ", out["category"], "/", out["priority"])
print("expected:", case["expected"], "\n")
for g in graders.CODE_GRADERS:
    print(f"  {g.__name__:<22} -> {g(case, out)}")

## 3. Metrics

`run_eval.summarize()` turns per-case grades into **per-grader**, **overall** (all graders pass, which is
strict, and what a user feels), and **per-tag** pass rates.

In [ ]:
records = run(dev)
s = report(records, label="baseline")
print("\nweakest tags:")
for tag, v in sorted(s["by_tag"].items(), key=lambda kv: kv[1]["pass_rate"])[:6]:
    print(f"  {tag:<12} n={v['n']}  {v['pass_rate']:.0%}")

## 4. Harness hygiene: errors count as failures

If you **skip** errored cases, a version that crashes on hard inputs looks *better*. Here's a task that
crashes on long tickets (a stand-in for a timeout or a malformed JSON response):

In [ ]:
def flaky_task(text, seed=0):
    # crashes on the hard inputs: very long, non-English, or SHOUTED text
    if len(text) > 150 or not text.isascii() or re.search(r"[A-Z]{5,}", text):
        raise TimeoutError("model timed out")
    return mock_task(text, seed=seed)

base = mean(r["all_passed"] for r in run(dev))
recs = run(dev, task=flaky_task)
errored = [r for r in recs if r["error"]]
print(f"{len(errored)} cases errored\n")

counted = mean(r["all_passed"] for r in recs)
skipped = mean(r["all_passed"] for r in recs if not r["error"])
print(f"baseline (never crashes):              {base:.0%}")
print(f"flaky version, errors counted as fails: {counted:.0%}   <- correctly worse")
print(f"flaky version, errors skipped:         {skipped:.0%}   <- looks BETTER than baseline. Wrong.")

### 🧠 Check
Your validator calls an LLM with a long Issue plus the whole Standard as context. Which Issues are most
likely to error or time out? What would skipping them do to your reported accuracy?

Other hygiene rules: **same inputs every time**, **save raw outputs** (not just scores) so you can re-grade,
and **record the config** (prompt version, model, date).

## 5. ✍️ Your turn: design an Issue case

Write **three** cases for your validator in the same shape as `tickets.jsonl`. Use made-up or anonymised
content. Decide what `input` must contain (what does the validator *see*?) and what `expected` is.

The rating levels below are **placeholders**. Replace them with your Standard's real scale.

In [ ]:
RATINGS = ["Low", "Medium", "High", "Very High"]   # placeholder: use your Standard's levels

my_issue_cases = [
    # ✍️ Example shape, replace with your own:
    # {
    #   "id": "i001",
    #   "input": {"title": "...", "description": "...", "operational_impact": "...",
    #             "assigned_rating": "Medium"},
    #   "expected": {"rating": "High", "acceptable": {"rating": []},
    #                "verdict": "under-rated", "standard_clause": "..."},
    #   "tags": ["borderline", "third-party"],
    #   "notes": "why this case exists",
    # },
]
print(f"{len(my_issue_cases)} cases written")

### 🧠 Check
Your validator's output could be **(a)** the correct rating, or **(b)** a verdict like "assigned rating is
correct / under-rated / over-rated". Which is easier to grade? Which is more useful to the user? Could you do both?

---
## As a lead engineer…

When someone shows you an eval result, ask about all four parts:
1. **Dataset:** how many cases, where from, who labelled them?
2. **Task:** is this *exactly* the production prompt + context + model?
3. **Grader:** how do we know the grader is right?
4. **Metric:** what's the error bar, and what are the failures?

### 🧠 Discussion
For the validator, who on your team or in 2nd-line risk could answer question 1 today?

**Recap**
- Dataset → task → grader → metric. Every eval has these four, and every weak eval is weak in one of them.
- Use single-purpose graders, slice by tags, and count errors as failures.
- Case design (what the validator sees, and what counts as "right") is a decision your team has to make.

**Next:** `/lab 1`, then `/lesson 3`.